У цьому ДЗ ми потренуємось розв'язувати задачу багатокласової класифікації за допомогою логістичної регресії з використанням стратегій One-vs-Rest та One-vs-One, оцінити якість моделей та порівняти стратегії.

### Опис задачі і даних

**Контекст**

В цьому ДЗ ми працюємо з даними про сегментацію клієнтів.

Сегментація клієнтів – це практика поділу бази клієнтів на групи індивідів, які схожі між собою за певними критеріями, що мають значення для маркетингу, такими як вік, стать, інтереси та звички у витратах.

Компанії, які використовують сегментацію клієнтів, виходять з того, що кожен клієнт є унікальним і що їхні маркетингові зусилля будуть більш ефективними, якщо вони орієнтуватимуться на конкретні, менші групи зі зверненнями, які ці споживачі вважатимуть доречними та які спонукатимуть їх до купівлі. Компанії також сподіваються отримати глибше розуміння уподобань та потреб своїх клієнтів з метою виявлення того, що кожен сегмент цінує найбільше, щоб точніше адаптувати маркетингові матеріали до цього сегменту.

**Зміст**.

Автомобільна компанія планує вийти на нові ринки зі своїми існуючими продуктами (P1, P2, P3, P4 і P5). Після інтенсивного маркетингового дослідження вони дійшли висновку, що поведінка нового ринку схожа на їхній існуючий ринок.

На своєму існуючому ринку команда з продажу класифікувала всіх клієнтів на 4 сегменти (A, B, C, D). Потім вони здійснювали сегментовані звернення та комунікацію з різними сегментами клієнтів. Ця стратегія працювала для них надзвичайно добре. Вони планують використати ту саму стратегію на нових ринках і визначили 2627 нових потенційних клієнтів.

Ви маєте допомогти менеджеру передбачити правильну групу для нових клієнтів.

В цьому ДЗ використовуємо дані `customer_segmentation_train.csv`[скачати дані](https://drive.google.com/file/d/1VU1y2EwaHkVfr5RZ1U4MPWjeflAusK3w/view?usp=sharing). Це `train.csv`з цього [змагання](https://www.kaggle.com/datasets/abisheksudarshan/customer-segmentation/data?select=train.csv)

**Завдання 1.** Завантажте та підготуйте датасет до аналізу. Виконайте обробку пропущених значень та необхідне кодування категоріальних ознак. Розбийте на тренувальну і тестувальну вибірку, де в тесті 20%. Памʼятаємо, що весь препроцесинг ліпше все ж тренувати на тренувальній вибірці і на тестувальній лише використовувати вже натреновані трансформери.
Але в даному випадку оскільки значень в категоріях небагато, можна зробити обробку і на оригінальних даних, а потім розбити - це простіше. Можна також реалізувати процесинг і тренування моделі з пайплайнами. Обирайте як вам зручніше.

In [52]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report, f1_score
from imblearn.over_sampling import SMOTENC
from imblearn.combine import SMOTETomek

In [6]:
df = pd.read_csv('customer_segmentation_train.csv', index_col=0)

In [7]:
df.dtypes

Gender              object
Ever_Married        object
Age                  int64
Graduated           object
Profession          object
Work_Experience    float64
Spending_Score      object
Family_Size        float64
Var_1               object
Segmentation        object
dtype: object

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8068 entries, 462809 to 461879
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Gender           8068 non-null   object 
 1   Ever_Married     7928 non-null   object 
 2   Age              8068 non-null   int64  
 3   Graduated        7990 non-null   object 
 4   Profession       7944 non-null   object 
 5   Work_Experience  7239 non-null   float64
 6   Spending_Score   8068 non-null   object 
 7   Family_Size      7733 non-null   float64
 8   Var_1            7992 non-null   object 
 9   Segmentation     8068 non-null   object 
dtypes: float64(2), int64(1), object(7)
memory usage: 693.3+ KB


In [36]:
print(df.isna().sum())

Gender             0
Ever_Married       0
Age                0
Graduated          0
Profession         0
Work_Experience    0
Spending_Score     0
Family_Size        0
Var_1              0
Segmentation       0
dtype: int64


In [17]:
df = df.dropna(subset=['Ever_Married', 'Graduated', 'Profession', 'Var_1', 'Work_Experience', 'Family_Size'])

Обробка пропущених значень: з колонок 'Ever_Married', 'Graduated', 'Profession', 'Var_1', 'Work_Experience', 'Family_Size' видалила пропущені значення і загалом кількість значень зменшилася на 18%



In [39]:
X = df.drop(columns=['Segmentation'])
y = df['Segmentation']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [40]:
# Pipeline препроцесингу (fit тільки на train, transform на train і test)
# Бінарні: Gender, Ever_Married, Graduated
# Ординальна: Spending_Score (Low < Average < High)
# Номінальні: Profession, Var_1 — One-Hot
# Числові: Age, Work_Experience, Family_Size — StandardScaler (масштабування для логрегресії!)

preprocessor = ColumnTransformer([
    ('binary', OrdinalEncoder(categories=[['Female', 'Male'], ['No', 'Yes'], ['No', 'Yes']]), 
     ['Gender', 'Ever_Married', 'Graduated']),
    ('ordinal', OrdinalEncoder(categories=[['Low', 'Average', 'High']]), ['Spending_Score']),
    ('onehot', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), 
     ['Profession', 'Var_1']),
    ('numeric', StandardScaler(), ['Age', 'Work_Experience', 'Family_Size'])
], remainder='drop')

In [41]:
# Тренуємо препроцесор на тренувальній вибірці
preprocessor.fit(X_train)

# Застосовуємо до train і test (тільки transform — не fit!)
X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Train shape:", X_train_processed.shape)
print("Test shape:", X_test_processed.shape)

Train shape: (5332, 21)
Test shape: (1333, 21)


In [42]:
# Кодування цільової змінної (A, B, C, D → 0, 1, 2, 3)
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test) 

**Завдання 2. Важливо уважно прочитати все формулювання цього завдання до кінця!**

Застосуйте методи ресемплингу даних SMOTE та SMOTE-Tomek з бібліотеки imbalanced-learn до тренувальної вибірки. В результаті у Вас має вийти 2 тренувальних набори: з апсемплингом зі SMOTE, та з ресамплингом з SMOTE-Tomek.

Увага! В нашому наборі даних є як категоріальні дані, так і звичайні числові. Базовий SMOTE не буде правильно працювати з категоріальними даними, але є його модифікація, яка буде. Тому в цього завдання є 2 виконання

  1. Застосувати SMOTE базовий лише на НЕкатегоріальних ознаках.

  2. Переглянути інформацію про метод [SMOTENC](https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.SMOTENC.html#imblearn.over_sampling.SMOTENC) і використати цей метод в цій задачі. За цей спосіб буде +3 бали за це завдання і він рекомендований для виконання.

  **Підказка**: аби скористатись SMOTENC треба створити змінну, яка містить індекси ознак, які є категоріальними (їх номер серед колонок) і передати при ініціації екземпляра класу `SMOTENC(..., categorical_features=cat_feature_indeces)`.
  
  Ви також можете розглянути варіант використання варіації SMOTE, який працює ЛИШЕ з категоріальними ознаками [SMOTEN](https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.SMOTEN.html)

In [48]:
# Завдання 2: SMOTENC (робота з категоріальними + числовими ознаками)
# Індекси категоріальних колонок у X_train_processed:
# 0-2: binary (Gender, Ever_Married, Graduated), 3: ordinal (Spending_Score), 4-17: one-hot (Profession, Var_1)
# 18-20: числові (Age, Work_Experience, Family_Size)
cat_feature_indices = list(range(18))

# 1) SMOTE 
smote_nc = SMOTENC(categorical_features=cat_feature_indices, random_state=42, k_neighbors=5)
X_train_smote, y_train_smote = smote_nc.fit_resample(X_train_processed, y_train_encoded)

# 2) SMOTE-Tomek 
smote_tomek = SMOTETomek(
    smote=SMOTENC(categorical_features=cat_feature_indices, random_state=42, k_neighbors=5),
    random_state=42
)
X_train_smote_tomek, y_train_smote_tomek = smote_tomek.fit_resample(X_train_processed, y_train_encoded)

print("Оригінал:", X_train_processed.shape[0], "зразків")
print("SMOTE:", X_train_smote.shape[0], "зразків")
print("SMOTE-Tomek:", X_train_smote_tomek.shape[0], "зразків")
# Розподіл класів (A, B, C, D)
print("\nРозподіл після SMOTE:", dict(zip(le.classes_, np.bincount(y_train_smote, minlength=4))))
print("Розподіл після SMOTE-Tomek:", dict(zip(le.classes_, np.bincount(y_train_smote_tomek, minlength=4))))

Оригінал: 5332 зразків
SMOTE: 5624 зразків
SMOTE-Tomek: 4356 зразків

Розподіл після SMOTE: {'A': np.int64(1406), 'B': np.int64(1406), 'C': np.int64(1406), 'D': np.int64(1406)}
Розподіл після SMOTE-Tomek: {'A': np.int64(1061), 'B': np.int64(1056), 'C': np.int64(1101), 'D': np.int64(1138)}


**Завдання 3**.
  1. Навчіть модель логістичної регресії з використанням стратегії One-vs-Rest з логістичною регресією на оригінальних даних, збалансованих з SMOTE, збалансованих з Smote-Tomek.  
  2. Виміряйте якість кожної з натренованих моделей використовуючи `sklearn.metrics.classification_report`.
  3. Напишіть, яку метрику ви обрали для порівняння моделей.
  4. Яка модель найкраща?
  5. Якщо немає суттєвої різниці між моделями - напишіть свою гіпотезу, чому?

In [56]:
# 1) Модель на оригінальних даних
lr_original = OneVsRestClassifier(LogisticRegression(max_iter=1000, random_state=42))
lr_original.fit(X_train_processed, y_train_encoded)
y_pred_original = lr_original.predict(X_test_processed)

# 2) Модель на SMOTE
lr_smote = OneVsRestClassifier(LogisticRegression(max_iter=1000, random_state=42))
lr_smote.fit(X_train_smote, y_train_smote)
y_pred_smote = lr_smote.predict(X_test_processed)

# 3) Модель на SMOTE-Tomek
lr_smote_tomek = OneVsRestClassifier(LogisticRegression(max_iter=1000, random_state=42))
lr_smote_tomek.fit(X_train_smote_tomek, y_train_smote_tomek)
y_pred_smote_tomek = lr_smote_tomek.predict(X_test_processed)

In [54]:
# classification_report для кожної моделі (target_names = A, B, C, D)
print("=" * 60)
print("1. Оригінальні дані")
print("=" * 60)
print(classification_report(y_test_encoded, y_pred_original, target_names=le.classes_, zero_division=0))

print("=" * 60)
print("2. SMOTE")
print("=" * 60)
print(classification_report(y_test_encoded, y_pred_smote, target_names=le.classes_, zero_division=0))

print("=" * 60)
print("3. SMOTE-Tomek")
print("=" * 60)
print(classification_report(y_test_encoded, y_pred_smote_tomek, target_names=le.classes_, zero_division=0))

1. Оригінальні дані
              precision    recall  f1-score   support

           A       0.47      0.54      0.50       323
           B       0.36      0.15      0.21       315
           C       0.49      0.66      0.56       344
           D       0.64      0.69      0.66       351

    accuracy                           0.52      1333
   macro avg       0.49      0.51      0.49      1333
weighted avg       0.50      0.52      0.49      1333

2. SMOTE
              precision    recall  f1-score   support

           A       0.46      0.56      0.51       323
           B       0.35      0.19      0.24       315
           C       0.52      0.62      0.57       344
           D       0.65      0.66      0.66       351

    accuracy                           0.52      1333
   macro avg       0.50      0.51      0.49      1333
weighted avg       0.50      0.52      0.50      1333

3. SMOTE-Tomek
              precision    recall  f1-score   support

           A       0.44      0.

In [ ]:
# Порівняння моделей за метрикою macro F1 
f1_original = f1_score(y_test_encoded, y_pred_original, average='macro', zero_division=0)
f1_smote = f1_score(y_test_encoded, y_pred_smote, average='macro', zero_division=0)
f1_smote_tomek = f1_score(y_test_encoded, y_pred_smote_tomek, average='macro', zero_division=0)

print("Macro F1 (метрика порівняння):")
print(f"  Оригінал:     {f1_original:.4f}")
print(f"  SMOTE:        {f1_smote:.4f}")
print(f"  SMOTE-Tomek:  {f1_smote_tomek:.4f}")

best_idx = np.argmax([f1_original, f1_smote, f1_smote_tomek])
best_name = ["Оригінал", "SMOTE", "SMOTE-Tomek"][best_idx]
print(f"\nНайкраща модель: {best_name}")

Macro F1 (метрика порівняння):
  Оригінал:     0.4854
  SMOTE:        0.4937
  SMOTE-Tomek:  0.4782

Найкраща модель: SMOTE


### Відповіді на Завдання 3

**3. Метрика порівняння:** Macro F1 — усереднена F1 по всіх класах без врахування їх розміру. Підходить для незбалансованої багатокласової класифікації, бо не переважає великі класи.

**4. Найкраща модель:** SMOTE (macro F1 = 0.4937). Оригінал — 0.4854, SMOTE-Tomek — 0.4782. Різниця невелика.

**5. Якщо різниця несуттєва — гіпотеза:** Різниця між моделями мала (~0.01). Можливі причини: (а) класи A, B, C, D досить збалансовані (≈1572–1757 зразків), тому ресемплинг дає невеликий ефект; (б) синтетичні зразки SMOTE не додають суттєвої нової інформації; (в) Tomek links видаляє багато зразків (4356 vs 5624) і може втратити корисну інформацію.

Пункт 4: Найкраща модель — SMOTE (macro F1 = 0.4937), за нею оригінал (0.4854) і SMOTE-Tomek (0.4782).

Пункт 5: Зафіксовано невелику різницю (~0.01)

Головна гіпотеза:
Класи A, B, C, D достатньо збалансовані (≈25% кожен), тому ресемплинг (SMOTE, SMOTE-Tomek) мало змінює поведінку моделі. Він має сенс при сильному дисбалансі, а в цьому випадку модель і на оригінальних даних справляється досить добре.
